In [ ]:
!pip install numpy pandas sqlalchemy psycopg2-binary tqdm matplotlib

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
username = 'BP_USER'
password = 'BP_PASS'
host = '192.168.178.22'
port = '5432'
database = 'BP_DB'
connection = create_engine(F"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

In [ ]:
bottom = -70 # 20 # -23.5
top = 30 # 10 # -3.5
left = -10 # 20 # -91.8
right = 160 # 10 # 71.8
start = '2020-01-01' # min: 2012-01-01
end = '2020-12-31' # max: 2020-12-31
timeframe = 'day'
# and cell_ll_lat between '{bottom}' and '{top}' 
# and cell_ll_lon between '{left}' and '{right}'
query = f"""
select "date", TO_DATE(to_char(date_trunc('{timeframe}', TO_DATE("date", 'YYYY-MM-DD')),'YYYYMMDD'),'YYYYMMDD') 
- TO_DATE('2012-01-01', 'YYYY-MM-DD') + 1 as time, sum(fishing_hours) as tfh,
cell_ll_lon as lon, cell_ll_lat as lat
from "FishingHours" fh
where "date" between '{start}' and '{end}'
group by date_trunc('{timeframe}', TO_DATE("date", 'YYYY-MM-DD')), "date", cell_ll_lat, cell_ll_lon
order by time asc
"""
print(query)

In [ ]:
df = pd.read_sql(query, connection)
df

In [5]:
df['i'] = df.index
# normalize the data 
df['x'] = (df['lon'] - df['lon'].min()) / (df['lon'].max() - df['lat'].min())
df['y'] = (df['lat'] - df['lat'].min()) / (df['lat'].max() - df['lat'].min())
# transform to numpy array
data = df.loc[:, ['time','x','y','i']].values #.to_numpy(dtype=np.float16)

In [6]:
def st_dbscan(data, eps1, eps2, min_points):
    labels = [None] * len(data)         # initializing an empty label array
    c = -1                              # initialize cluster counter
    for p in tqdm(data):                      # iterating over every point in the cluster
        i = int(p[3])                   # index of the current point
        if labels[i] is not None:       # determine if p has been processed
            continue
        N = spatialNeighbors(temporalNeighbors(data, eps2, p), eps1, p) # neighborhood
        if len(N) < min_points:         # determine neighborhood size meets min_points
            labels[i] = -1              # label point as noise
            continue
        c += 1                          # start next cluster
        labels[i] = c                   # set cluster label to this node
        S = N[N[:, 3] != p[3]].tolist() # S := N \ {P}
        while S:
            elem = S.pop()
            j = int(elem[3])            # get index of q in Data
            if labels[j] == -1:         # relabel noise point as neighbor
                labels[j] = c
            if labels[j] is not None:   # skip previously processed point
                continue
            labels[j] = c               # label current point
            N = spatialNeighbors(temporalNeighbors(data, eps2, elem), eps1, elem)
            if len(N) < min_points:
                continue
            # S = S union N
            S.extend(N)
    return np.array(labels)


def spatialNeighbors(data, eps, q):
    return data[np.sqrt(np.square(data[:, 1] - q[1]) + np.square(data[:, 2] - q[2])) <= eps]

def temporalNeighbors(data, eps, q):
    return data[abs(data[:, 0] - q[0]) < eps] 

In [ ]:
# clusters = st_dbscan(data, 0.00385, 14, 10)

In [ ]:
# len(set(clusters))

In [24]:
def plotDays2020(data, labels, save = False, filename='clust'):
    #         January  , Febuary , March   , April   , May     , June    , July    , August    , September, October , November , December
    #         Green    , Red,    , tan     , Orange  , Skin    , lilac  , Purple   , Grey-Blue , Maroon  , Sky-Blue , Pale-Blue, Pale-Green
    colors = ['#33a02c','#e31a1c','#fdbf6f','#ff7f00','#fb9a99','#cab2d6','#6a3d9a', '#606E8C' ,'#5E2129', '#a6cee3', '#1f78b4', '#b2df8a']
    cI = 0
    labelset = set(labels)
    labelList = list(labelset)
    xPlots = 4
    yPlots = int(len(labelset)/xPlots)
    oPlots = len(labelset) % xPlots
    
    if oPlots > 0:
        yPlots += 1
    
    fig, axs = plt.subplots(yPlots, xPlots, sharex=True, sharey=True, figsize=(20, 200)) # , figsize=(20, 100)
    for i in tqdm(range(0, yPlots)):
        for j in range(0, xPlots):
            clust = data[np.where(labels==labelList[cI])]
            for k in range(0, len(colors)-1):
                col = colors[k]

                if k == 0:
                    time = clust[np.where((clust[:,0] <= 2953) & (clust[:,0] >= 2923))]
                elif k == 1:
                    time = clust[np.where((clust[:,0] <= 2982) & (clust[:,0] >= 2954))]
                # elif k == 2:
                #     time = clust[np.where((clust[:,0] <= 3013) & (clust[:,0] >= 2983))]
                # elif k == 3:
                #     time = clust[np.where((clust[:,0] <= 3043) & (clust[:,0] >= 3014))]
                # elif k == 4:
                #     time = clust[np.where((clust[:,0] <= 3074) & (clust[:,0] >= 3044))]
                # elif k == 5:
                #     time = clust[np.where((clust[:,0] <= 3104) & (clust[:,0] >= 3075))]
                # elif k == 6:
                #     time = clust[np.where((clust[:,0] <= 3135) & (clust[:,0] >= 3105))]
                # elif k == 7:
                #     time = clust[np.where((clust[:,0] <= 3166) & (clust[:,0] >= 3136))]
                # elif k == 8:
                #     time = clust[np.where((clust[:,0] <= 3196) & (clust[:,0] >= 3167))]
                # elif k == 9:
                #     time = clust[np.where((clust[:,0] <= 3227) & (clust[:,0] >= 3197))]
                # elif k == 10:
                #     time = clust[np.where((clust[:,0] <= 3257) & (clust[:,0] >= 3228))]
                # elif k == 11:
                #     time = clust[np.where((clust[:,0] <= 3288) & (clust[:,0] >= 3258))]
    
                axs[i,j].scatter(time[:,1], time[:,2], c=[col], s=1)
            axs[i,j].set_title(f"cluster ID:{labelList[cI]}")
            
            if len(labelList) - 1 > cI:
                cI += 1
    if save:
        plt.savefig(filename, dpi=300)
    
    plt.show()
    return fig, axs

In [28]:
# plotDays2020(data, clusters, False, 'clust_0.05_7_10')

In [6]:
import numpy as np
from tqdm import tqdm
import math

def st_dbscan(data, eps1, eps2, min_points):
    num_points = len(data)
    labels = np.full(num_points, None, dtype=object)
    c = -1

    # Determine grid size based on eps1 and eps2
    spatial_cell_size = eps1
    temporal_cell_size = eps2

    # Compute the bounds of the data
    min_time = data[:, 0].min()
    min_x = data[:, 1].min()
    min_y = data[:, 2].min()

    # Assign each point to a cell
    temporal_indices = ((data[:, 0] - min_time) / temporal_cell_size).astype(int)
    x_indices = ((data[:, 1] - min_x) / spatial_cell_size).astype(int)
    y_indices = ((data[:, 2] - min_y) / spatial_cell_size).astype(int)

    # Create a mapping from cell indices to point indices
    grid = {}
    for idx in range(num_points):
        key = (temporal_indices[idx], x_indices[idx], y_indices[idx])
        if key not in grid:
            grid[key] = []
        grid[key].append(idx)

    # Helper function to get neighboring cells
    def get_neighboring_cells(t_idx, x_idx, y_idx):
        neighboring_cells = []
        for dt in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    neighbor_key = (t_idx + dt, x_idx + dx, y_idx + dy)
                    if neighbor_key in grid:
                        neighboring_cells.append(neighbor_key)
        return neighboring_cells

    # Main clustering loop
    for idx in tqdm(range(num_points)):
        if labels[idx] is not None:
            continue

        # Get cell indices for the current point
        t_idx = temporal_indices[idx]
        x_idx = x_indices[idx]
        y_idx = y_indices[idx]

        # Get neighboring cells
        neighboring_cells = get_neighboring_cells(t_idx, x_idx, y_idx)

        # Collect neighbor indices
        neighbor_indices = []
        for cell_key in neighboring_cells:
            neighbor_indices.extend(grid[cell_key])

        # Remove the current point from neighbor_indices
        neighbor_indices = [n_idx for n_idx in neighbor_indices if n_idx != idx]

        # Apply distance conditions
        if neighbor_indices:
            neighbor_points = data[neighbor_indices]
            spatial_distances = np.sqrt((neighbor_points[:, 1] - data[idx, 1])**2 +
                                        (neighbor_points[:, 2] - data[idx, 2])**2)
            temporal_distances = np.abs(neighbor_points[:, 0] - data[idx, 0])

            # Filter neighbors based on eps1 and eps2
            valid_indices = [neighbor_indices[i] for i in range(len(neighbor_indices))
                             if spatial_distances[i] <= eps1 and temporal_distances[i] < eps2]

            if len(valid_indices) < min_points:
                labels[idx] = -1  # Mark as noise
                continue

            c += 1  # Start a new cluster
            labels[idx] = c
            seeds = set(valid_indices)

            while seeds:
                current_point = seeds.pop()
                if labels[current_point] == -1:
                    labels[current_point] = c
                if labels[current_point] is not None:
                    continue
                labels[current_point] = c

                # Get cell indices for the current point
                t_idx_cp = temporal_indices[current_point]
                x_idx_cp = x_indices[current_point]
                y_idx_cp = y_indices[current_point]

                # Get neighboring cells for the current point
                neighboring_cells_cp = get_neighboring_cells(t_idx_cp, x_idx_cp, y_idx_cp)

                # Collect neighbor indices for the current point
                neighbor_indices_cp = []
                for cell_key_cp in neighboring_cells_cp:
                    neighbor_indices_cp.extend(grid[cell_key_cp])

                # Remove the current point from neighbor_indices_cp
                neighbor_indices_cp = [n_idx for n_idx in neighbor_indices_cp if n_idx != current_point]

                if neighbor_indices_cp:
                    neighbor_points_cp = data[neighbor_indices_cp]
                    spatial_distances_cp = np.sqrt((neighbor_points_cp[:, 1] - data[current_point, 1])**2 +
                                                   (neighbor_points_cp[:, 2] - data[current_point, 2])**2)
                    temporal_distances_cp = np.abs(neighbor_points_cp[:, 0] - data[current_point, 0])

                    # Filter neighbors based on eps1 and eps2
                    valid_indices_cp = [neighbor_indices_cp[i] for i in range(len(neighbor_indices_cp))
                                        if spatial_distances_cp[i] <= eps1 and temporal_distances_cp[i] < eps2]

                    if len(valid_indices_cp) >= min_points:
                        seeds.update(valid_indices_cp)

    return np.array(labels)

In [ ]:

def st_dbscan(data, eps1, eps2, min_points):
    labels = np.full(len(data), None)  # Initialize labels array with None
    c = -1                            # Cluster counter
    
    for p in data:
        i = int(p[3])                 # Index of the current point
        
        if labels[i] is not None:     # Skip already processed points
            continue
            
        # Compute temporal neighbors using vectorized operations
        temp_mask = np.abs(data[:,0] - p[0]) < eps2
        temporal_neighbors = data[temp_mask]
        
        # Compute spatial neighbors among temporal neighbors
        dist_sq = (temporal_neighbors[:,1] - p[1])*2 + (temporal_neighbors[:,2] - p[2])*2
        spat_mask = dist_sq <= eps1**2
        N = temporal_neighbors[spat_mask]
        
        if len(N) < min_points:      # Check if neighborhood size meets min_points
            labels[i] = -1            # Label point as noise
            continue
        
        c += 1                       # Start next cluster
        labels[i] = c                # Set cluster label to this node
        
        S = N[N[:,3] != p[3]].tolist()  # Remove the current point from neighborhood
        
        while S:
            elem = S.pop(0)          # Pop from start for FIFO order
            
            j = int(elem[3])         # Index of q in Data
            if labels[j] == -1:      # Relabel noise point as neighbor
                labels[j] = c
                
            if labels[j] is not None: # Skip previously processed points
                continue
                
            labels[j] = c             # Label current point
            
            # Compute temporal neighbors for elem using vectorized operations
            temp_mask_elem = np.abs(data[:,0] - elem[0]) < eps2
            temporal_neighbors_elem = data[temp_mask_elem]
            
            # Compute spatial neighbors among temporal neighbors of elem
            dist_sq_elem = (temporal_neighbors_elem[:,1] - elem[1])*2 + (temporal_neighbors_elem[:,2] - elem[2])*2
            spat_mask_elem = dist_sq_elem <= eps1**2
            N_elem = temporal_neighbors_elem[spat_mask_elem]
            
            if len(N_elem) < min_points:
                continue
                
            # Extend S with new neighbors
            S.extend(N_elem)
    
    return labels

In [ ]:
clusters = st_dbscan(data, 0.00365, 7, 10)

In [ ]:
len(set(clusters))

In [38]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def plotDays2020_parallel(data, labels, save=False, filename='clust'):
    # Define colors
    colors = ['#33a02c','#e31a1c','#fdbf6f','#ff7f00','#fb9a99','#cab2d6','#6a3d9a',
              '#606E8C','#5E2129','#a6cee3','#1f78b4','#b2df8a']
    
    labelset = set(labels)
    labelList = list(labelset)
    xPlots = 4
    yPlots = len(labelList) // xPlots + (len(labelList) % xPlots > 0)
    
    fig, axs = plt.subplots(yPlots, xPlots, sharex=True, sharey=True, figsize=(20, 200))
    results = {}

    # Predefine time ranges for each color index
    time_ranges = [
        (2923, 2953),
        (2954, 2982),
        # Add more ranges if needed
    ]

    def process_cluster(label):
        clust = data[labels == label]
        plot_data = []
        for k, (start, end) in enumerate(time_ranges):
            col = colors[k]
            time = clust[(clust[:, 0] >= start) & (clust[:, 0] <= end)]
            plot_data.append((time[:, 1], time[:, 2], col))
        return label, plot_data

    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor() as executor:
        futures = {executor.submit(process_cluster, label): label for label in labelList}
        for future in tqdm(as_completed(futures), total=len(futures)):
            label, plot_data = future.result()
            results[label] = plot_data

    # Plot the results in the main thread
    cI = 0
    for i in range(yPlots):
        for j in range(xPlots):
            if cI >= len(labelList):
                break
            label = labelList[cI]
            plot_data = results[label]
            for x, y, col in plot_data:
                axs[i, j].scatter(x, y, c=[col], s=1)
            axs[i, j].set_title(f"Cluster ID: {label}")
            cI += 1

    if save:
        plt.savefig(filename, dpi=300)
    plt.show()
    return fig, axs


In [ ]:
# plotDays2020(data, clusters, False, 'clust_0.00385_7_10')

In [ ]:
df['cid'] = clusters
df.drop(columns=['i', 'time', 'x', 'y'], inplace=True)
df

In [ ]:
df.to_sql('cluster_data', connection, if_exists='replace')